# 03. MRI Brain Tumor Classification

Extract grayscale 64x64 image features that match the backend MRI preprocessor, then train a PCA + SVM pipeline and save the artifacts.


In [1]:
from pathlib import Path
import sys

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "NeuroSense" / "webdev" / "backend").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root for this notebook.")

PROJECT_ROOT = _find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "NeuroSense" / "notebooks"
BACKEND_DIR = PROJECT_ROOT / "NeuroSense" / "webdev" / "backend"

for path in (NOTEBOOKS_DIR, BACKEND_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook(PROJECT_ROOT)
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets directory: {DATASETS_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")


Project root: /Users/devashishsingh/Desktop/human emotion recognition system
Datasets directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/datasets
Artifacts directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts


In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from notebook_support import collect_labeled_image_paths, extract_feature_dataset, label_counts, resolve_mri_directories
from utils.preprocessors import preprocess_mri_image

def normalize_mri_label(folder_name):
    return "no_tumor" if folder_name.lower() == "notumor" else folder_name.lower()

train_dir, test_dir = resolve_mri_directories(DATASETS_DIR)
train_records = collect_labeled_image_paths(train_dir, normalize_mri_label)
test_records = collect_labeled_image_paths(test_dir, normalize_mri_label)

print("MRI training labels:", label_counts(label for _, label in train_records))
print("MRI testing labels:", label_counts(label for _, label in test_records))

X_train_raw, y_train_raw = extract_feature_dataset(
    train_records,
    preprocess_mri_image,
    cache_path=CACHE_DIR / "mri_train_features.npz",
    progress_interval=250,
)
X_test_raw, y_test_raw = extract_feature_dataset(
    test_records,
    preprocess_mri_image,
    cache_path=CACHE_DIR / "mri_test_features.npz",
    progress_interval=250,
)

print("MRI feature matrix:", X_train_raw.shape, X_test_raw.shape)


MRI training labels: {'glioma': 1400, 'meningioma': 1400, 'no_tumor': 1400, 'pituitary': 1400}
MRI testing labels: {'glioma': 400, 'meningioma': 400, 'no_tumor': 400, 'pituitary': 400}


MRI feature matrix: (5600, 4096) (1600, 4096)


In [3]:
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_test_sc = scaler.transform(X_test_raw)

pca = PCA(n_components=100, svd_solver="randomized", random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca = pca.transform(X_test_sc)

model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)

print(f"MRI test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("PCA explained variance:", round(float(np.sum(pca.explained_variance_ratio_)), 4))


/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:341: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:341: RuntimeWarning: overflow encountered in matmul
  Q, _ =

/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:345: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = qr_normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:345: RuntimeWarning: overflow encountered in matmul
  Q, _ = qr_normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:345: RuntimeWarning: invalid value encountered in matmul
  Q, _ = qr_normalizer(A @ Q)


/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:532: RuntimeWarning: divide by zero encountered in matmul
  B = Q.T @ M
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:532: RuntimeWarning: overflow encountered in matmul
  B = Q.T @ M
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:532: RuntimeWarning: invalid value encountered in matmul
  B = Q.T @ M
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:546: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:546: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/devashishsingh/Desktop/human emotion re

/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


MRI test accuracy: 0.8681
              precision    recall  f1-score   support

      glioma       0.84      0.70      0.77       400
  meningioma       0.84      0.82      0.83       400
    no_tumor       0.86      1.00      0.92       400
   pituitary       0.93      0.94      0.94       400

    accuracy                           0.87      1600
   macro avg       0.87      0.87      0.86      1600
weighted avg       0.87      0.87      0.86      1600

PCA explained variance: 0.7054


In [4]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("MRI confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

figure, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (sample, label) in zip(axes, zip(X_test_raw[:3], y_test_raw[:3])):
    axis.imshow(sample.reshape(64, 64), cmap="gray")
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()
plt.show()


/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59784/2932075145.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59784/2932075145.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
try:
    import torch
    import torch.nn as nn
    from torchvision import models

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cnn_model = models.resnet18(weights=None)
    cnn_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    cnn_model.fc = nn.Linear(cnn_model.fc.in_features, len(le.classes_))
    cnn_model = cnn_model.to(device)
    print("Optional CNN baseline instantiated on:", device)
except Exception as exc:
    print(f"Skipping optional CNN baseline: {exc}")


Skipping optional CNN baseline: No module named 'torch'


In [6]:
artifact_dir = ARTIFACTS_DIR / "mri"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "mri_model.pkl")
joblib.dump(scaler, artifact_dir / "mri_scaler.pkl")
joblib.dump(pca, artifact_dir / "mri_pca.pkl")
joblib.dump(le, artifact_dir / "mri_label_encoder.pkl")

print("Saved MRI artifacts to:", artifact_dir)


Saved MRI artifacts to: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts/mri
